# Transformer VQ-VAE optimization via Optuna

In [2]:
import optuna
from optuna.integration import PyTorchLightningPruningCallback
import lightning as L
import matplotlib.pyplot as plt
import torch
import yaml

from src.Data.data_loading import ParquetDataModule
from src.Data.data_loading import ParquetFeatureDataset
from src.Models.VQ_VAE_transformer import VQVAE_Transformer

In [3]:
#Open config file
with open("Configs/config.yaml") as f:
    config = yaml.safe_load(f)

#Extract config info
#Directories
dirs_train = config["data"]["train_path"]
dirs_val = config["data"]["val_path"]
dirs_test = config["data"]["test_path"]

#Features
features_cols = config["data"]["features"]
max_part = config["data"]["max_part"]
prep = config["data"]["preprocessing"]

#Batch size
batch_size = config["training"]["batch_size"]

#Rotation trick
rot_trick = config["Transformer_VQVAE"]["rotation_trick"]

#Initialization of the lightining datamodule 
data_module = ParquetDataModule(
    parquet_dirs_train=dirs_train, 
    parquet_dirs_val=dirs_val,
    parquet_dirs_test=dirs_test,
    features=features_cols,
    window_particles=max_part,
    batch_size=batch_size,
    preprocessing=prep
    #num_workers=0
)

In [4]:
def objective(trial):

    latent_dim = trial.suggest_categorical("latent_dim", [64, 128, 256])
    n_heads = trial.suggest_categorical("n_heads", [2, 4, 8])
    n_layers = trial.suggest_int("n_layers", 2, 6)

    if latent_dim % n_heads != 0:
        raise optuna.exceptions.TrialPruned()

    codebook_size = trial.suggest_categorical("codebook_size", [128, 256, 512])
    beta = trial.suggest_float("beta", 0.1, 1.0)
    decay = trial.suggest_float("decay", 0.7, 0.99)

    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    


    model = VQVAE_Transformer(
        input_dim=3,
        latent_dim=latent_dim,
        codebook_size=codebook_size,
        n_heads=n_heads,
        n_layers=n_layers,
        lr=lr,
        dec=decay,
        beta=beta,
        rot_trick=rot_trick
    )

    trainer = L.Trainer(
        max_epochs=20,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        callbacks=[PyTorchLightningPruningCallback(trial, monitor="val_loss")]
    )

    trainer.fit(model, datamodule=data_module)

    return trainer.callback_metrics["val_loss"].item()

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

[I 2026-04-29 21:30:38,427] A new study created in memory with name: no-name-b9020ece-a88d-42a4-a420-94dac9dc8489
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ TransformerEncoder │  5.3 M │ train │     0 │
│ 1 │ decoder │ TransformerDecoder │  5.3 M │ train │     0 │
│ 2 │ vq      │ VectorQuantize     │      0 │ train │     0 │
└───┴─────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 10.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.5 M                                                                                               
Total estimated model params size (MB): 42                                                                         
Modules in train mode: 92                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/francesco/anaconda3/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/francesco/anaconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
/home/francesco/anaconda3/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottlene

In [ ]:
print(study.best_params)